In [ ]:
import pandas as pd
import datetime as dt
import numpy as np
import matplotlib.pyplot as plt

## Total Returns Extended to Commodities

### Data sources
- st louis 3m tbill used as risk free rate for determination of sharpe, it can be used as approximation funding cost available since 1930
- shiller 10y bond rate and equity return and div, this allows computing 10y total return and equity total return
- spot commo prices from CMO, spot can be used approx for return on precious metals, and possibly copper. Others, notable oil spot price return is missing massive future roll return
- CL1 and CL6 historical data from bloomberg, since 1991. This allows computing future roll return, which is significant for oil.

In [ ]:
def tbillrate():
    df = pd.read_csv("data/TB3MS.csv")
    df["observation_date"] = pd.to_datetime(df["observation_date"])
    df["TB3MS"] /= 100
    return df.set_index("observation_date")
def compute_gold_tr(df):
    for colname in df.columns:
        df[colname+'TR'] = df[colname]/df[colname].shift(1)
    return df
def goldprice(col):
    df = pd.read_csv("data/cmo-data-monthly.csv")
    df = df[["date"]+list(col.keys())]
    df["date"] = pd.to_datetime(df["date"])+dt.timedelta(days=1)
    df = df.loc[df["date"]>=dt.datetime(1971,2,1)]
    return compute_gold_tr(df.set_index("date").rename(columns=col))
def read_shiller_out():
    df = pd.read_csv("data/shiller_out.csv")
    df['Date'] = pd.to_datetime(df['Date'], format="%Y-%m-%d")+dt.timedelta(days=-14) # imported as 14th, will be used as 1st 
    df = df.set_index("Date")
    df = df.join(tbillrate(),how="inner")
    compute_tbill_tr(df)
    return df
def read_shiller_cmo(col):
    return read_shiller_out().join(goldprice(col),how="inner")
          
"""
def compute_bond_tr(df):
    r = df['Rate10y']
    T = 10
    duration = -(1-np.exp(-r*T))/r
    bondcarry = r.shift(1)/12
    bondtotalret = 1+(r-r.shift(1))*duration+bondcarry
    df["bondTR"] = bondtotalret
def compute_eq_tr(df):
    df['eqTR'] = df['SP500']/df['SP500'].shift(1)+df['Div']/12/df['SP500']
def compute_cpi_tr(df):
    df['cpiTR'] = 1+df['CPI'].pct_change()
    """
def compute_tbill_tr(df):
    df['tbTR'] = 1+df['TB3MS'].shift(1)*365/360/12 # compute with shift with last rate, as if this was a 1m rate

def metrics_monthly_ret(df):
    trcolumns = [c for c in df.columns if "TR"==c[-2:] and c!="cpiTR"]
    logret = np.log(df[trcolumns].dropna())
    mu,sigma = logret.mean()*12,logret.std()*np.sqrt(12)
    #sigma["tbTR"] = 0
    mu += 0.5*sigma**2
    r = mu["tbTR"]
    riskycol = [c for c in trcolumns if c != 'tbTR']
    print(riskycol)
    mustar = np.expm1(mu[riskycol]-r)
    data = {'mustar':mustar,'sigma':sigma[riskycol]}
    data['sharpe'] = data['mustar']/data['sigma']
    corr = logret[riskycol].corr()
    cov = logret[riskycol].cov()*12
    wstar = np.linalg.solve(cov,mustar)
    K = np.abs(np.sum(wstar))
    data['w'] = wstar/K
    for c in riskycol:
        data[f'rho({c[:-2]})'] = corr[c]
    return pd.DataFrame(data,index=riskycol),r,K

def show_returns(df,filename,excl):
    cols = [c for c in df.columns if c[-2:]=="TR" and c not in excl]
    df = df[cols]
    dfmetrics,r,K = metrics_monthly_ret(df)
    print(dfmetrics.index)
    for c in dfmetrics.index:
        plt.plot(np.cumprod(df[c]/df["tbTR"]),
                label=f"{c[:-2]}: $\mu^*$={dfmetrics.loc[c,'mustar']:.1%}, $\sigma$={dfmetrics.loc[c,'sigma']:.0%}, S={dfmetrics.loc[c,'sharpe']:.2f}, w={dfmetrics.loc[c,'w']:.0%}")
    plt.legend()
    plt.title(f"Asset Total Return r={r:.1%} K={K:.1f}")
    plt.ylabel("log total return")
    plt.yscale('log')
    datesstr = f"from {str(df.index[0])[:10]} to {str(df.index[-1])[:10]}"
    plt.xlabel(datesstr)
    print(datesstr)
    print(f"r={r:.2%} K={K:.1f} (kelly leverage)")
    plt.grid(True)  
    if not filename is None:
        plt.savefig(filename)
        plt.close()
    else:
        plt.show()
    return dfmetrics

def show_all_returns(df,show,excl):
    assets = [c[:-2] for c in df.columns if "TR" in c and c not in ["cpiTR","tbTR"]]
    filename = None if show else "_".join(assets).replace(" (spot)","spot")+".png"
    dfmetrics = show_returns(df.loc[df.index>=dt.datetime(1800,1,1)],filename,excl)
    return dfmetrics,filename

def show_col_returns(col,show,excl):
    df = read_shiller_cmo(col)
    return show_all_returns(df,show,excl)



### Method
- tbill total return $\mu_r= 1+r . 365/360/12$ where $r$ is the rate in ACT.360 convention
- equity total return $\mu_e=P(i+1)/P(i) + D(i)/12$, where $P$ is price, $D$ annual dividend
- bond total return $\mu_b=A(i+1) (y(i+1)-y(i)) + y(i) / 12$ where $A$ is the annuity, $y$ the 10y yield
- commo price return $\mu_o=S(i+1)/S(i)$ where $S$ is spot price (this is total return for precious metals)
- contango amount $c=(F(i+6)-F(i+1))/F(i+6)$, contango rate $y_c=c^{1/5}$
- commo future total return $\mu_o=S(i+1)/S(i)-y_c$ 

### CMO Data
- crude oil spot shows big supply shocks in the 70s, return is significant
- gold, platinum, and silver are similar but gold has the lowest vol, 
- copper only recently started to beat inflation  from the 2000s (energiewende or China demand growing)
- softs tend to grow only 2%-3%, so return is flat above inflation, no excess return.
- algo prefers gold and copper to silver and platinum. The latter seems to be within efficient frontier of gold and copper.

### Metrics and Optimal Portfolio
- monthly log ret annualized vol $\sigma$
- monthly annualized expected return $\mu$ 
- monthly annualized expected excess return $\mu^*=\mu-r$
- Sharpe = $\mu^*/\sigma$
- optimal weightws $w^* = \Sigma^{-1} \mu^*$
- Optimal Kelly leverage $K=\sum(w)$
- normed weights $w = w^*/K$

## Shiller +Fed Data Only: Equity and Bond excess return from 1934 (92 years)

In [ ]:
dfsh = read_shiller_out()
show_all_returns(dfsh,True,[])



### Rate Strategy

In [ ]:


x = dfsh["TB3MS"]
y = dfsh["Rate10y"]
mask = ~(np.isnan(x) | np.isnan(y))
x_clean = x[mask].values
y_clean = y[mask].values
coeffs = np.polyfit(x_clean, y_clean, 1)
a, b = coeffs
y_pred = a * x_clean + b
errstd = np.std(y_clean - y_pred)
plt.scatter(x, y, alpha=0.2)
plt.plot(x_clean, np.polyval(coeffs, x_clean), 'r-', lw=2)
plt.text(0.05, 0.9, f'y = {coeffs[0]:.4f}x + {coeffs[1]:.4f} + {errstd:.4f} e', 
         transform=plt.gca().transAxes, bbox=dict(facecolor='white', alpha=0.7))
plt.xlabel("3M tbill rate")
plt.ylabel("10y rate")
plt.title("10y rate vs 3m rate")
plt.show()
x = (dfcl["TB3MS"]).shift(1)
y = dfcl["bondTR"]-dfcl["tbTR"]
mask = ~(np.isnan(x) | np.isnan(y))
x_clean = x[mask].values
y_clean = y[mask].values
coeffs = np.polyfit(x_clean, y_clean, 1)
a, b = coeffs
y_pred = a * x_clean + b
errstd = np.std(y_clean - y_pred)
plt.scatter(x, y, alpha=0.2)
plt.plot(x_clean, np.polyval(coeffs, x_clean), 'r-', lw=2)
plt.text(0.05, 0.9, f'y = {coeffs[0]:.4f}x + {coeffs[1]:.4f} + {errstd:.4f} e', 
         transform=plt.gca().transAxes, bbox=dict(facecolor='white', alpha=0.7))
plt.xlabel("3m rate")
plt.ylabel("bond net total return")
plt.title(f"bond net return vs 3m rate - pivot:{-b/a:.2%}")
plt.show()
print(a,b,errstd)
x = (dfcl["Rate10y"]-dfcl["TB3MS"]).shift(1)
y = dfcl["bondTR"]-dfcl["tbTR"]
mask = ~(np.isnan(x) | np.isnan(y))
x_clean = x[mask].values
y_clean = y[mask].values
coeffs = np.polyfit(x_clean, y_clean, 1)
a, b = coeffs
y_pred = a * x_clean + b
errstd = np.std(y_clean - y_pred)
plt.scatter(x, y, alpha=0.2)
plt.plot(x_clean, np.polyval(coeffs, x_clean), 'r-', lw=2)
plt.axvline(x=-b/a,color="black")
plt.text(0.05, 0.9, f'y = {coeffs[0]:.4f}x + {coeffs[1]:.4f} + {errstd:.4f} e', 
         transform=plt.gca().transAxes, bbox=dict(facecolor='white', alpha=0.7))
plt.xlabel("10y - 3m rate spread")
plt.ylabel("bond net total return")
plt.title(f"bond net return vs rate spread - pivot: {-b/a:.2%}")
plt.show()
print(a,b,errstd)


In [ ]:
x = (dfsh["bondTR"]-dfsh["tbTR"]).rolling(12).mean().shift(1)
y = dfsh["bondTR"]-dfsh["tbTR"]
mask = ~(np.isnan(x) | np.isnan(y))
x_clean = x[mask].values
y_clean = y[mask].values
coeffs = np.polyfit(x_clean, y_clean, 1)
a, b = coeffs
y_pred = a * x_clean + b
errstd = np.std(y_clean - y_pred)
plt.scatter(x, y, alpha=0.2)
plt.plot(x_clean, np.polyval(coeffs, x_clean), 'r-', lw=2)
plt.axvline(x=-b/a,color="black")
plt.text(0.05, 0.9, f'y = {coeffs[0]:.4f}x + {coeffs[1]:.4f} + {errstd:.4f} e', 
         transform=plt.gca().transAxes, bbox=dict(facecolor='white', alpha=0.7))
plt.xlabel("prev return")
plt.ylabel("bond net total return")
plt.title(f"bond net return vs rate spread - pivot: {-b/a:.2%}")
plt.show()
print(a,b,errstd)
print(dfsh.columns)
epsi = 0.02
bondcarrystrat = 1+(dfsh["bondTR"]-dfsh["tbTR"])*np.where(x > epsi, 1, np.where(x < -epsi, -1, 0))
dfsh["bondCarryTR"] = bondcarrystrat
show_all_returns(dfsh,True,[])


In [ ]:
import numpy as np
import pandas as pd

# 1. Compute the spread and the signal
spread = dfsh["bondTR"] - dfsh["tbTR"]
x = spread.rolling(12).mean().shift(1)   # your signal (lagged)

epsi = 0.02
# Use neutral weight = 0 (as you tried)
weight = np.where(x > epsi, 1, np.where(x < -epsi, -1, 0))

# 2. Strategy excess return (over t-bill)
strat_excess = weight * spread

# 3. Basic statistics on the full sample (including NaNs from rolling mean)
#    We'll use .dropna() to get the period where we actually have a signal.
valid = ~(spread.isna() | x.isna())
spread_valid = spread[valid]
weight_valid = weight[valid]
strat_valid = strat_excess[valid]

# 4. Compute monthly mean excess return
monthly_mean_strat = strat_valid.mean()
monthly_std_strat = strat_valid.std()
annualised_mean = monthly_mean_strat * 12
annualised_vol = monthly_std_strat * np.sqrt(12)
sharpe = annualised_mean / annualised_vol if annualised_vol != 0 else np.nan

# 5. Break down the active (non‑zero weight) periods
active = (weight_valid != 0)
active_frac = active.mean()

print("===== Full valid sample =====")
print(f"Number of valid months: {len(valid)}")
print(f"Mean spread (monthly): {spread_valid.mean():.8f}")
print(f"Mean weight: {weight_valid.mean():.6f}")
print(f"Mean excess return (strategy): {monthly_mean_strat:.8f}")
print(f"Annualised mean excess: {annualised_mean:.6f}")
print(f"Annualised vol: {annualised_vol:.6f}")
print(f"Sharpe: {sharpe:.4f}")

print("\n===== Active periods only (weight != 0) =====")
print(f"Fraction of active periods: {active_frac:.3%}")
print(f"Mean weight (active only): {weight_valid[active].mean():.2f}")
print(f"Mean spread (active only): {spread_valid[active].mean():.8f}")
print(f"Mean strategy excess (active only): {strat_valid[active].mean():.8f}")

print("\n===== Sign analysis (active only) =====")
same_sign = (weight_valid[active] * spread_valid[active]) > 0
opposite_sign = ~same_sign
print(f"Same sign fraction: {same_sign.mean():.1%}")
print(f"Avg PnL same sign: {strat_valid[active][same_sign].mean():.8f}")
print(f"Avg PnL opposite sign: {strat_valid[active][opposite_sign].mean():.8f}")
print(f"Avg |spread| same sign: {spread_valid[active][same_sign].abs().mean():.6f}")
print(f"Avg |spread| opposite sign: {spread_valid[active][opposite_sign].abs().mean():.6f}")

# 6. Compare with the summary table numbers
print("\n===== Comparison with your table =====")
print(f"Your table's mustar for bondCarryTR: -0.033783")
print(f"Computed annualised mean: {annualised_mean:.6f}")
if abs(annualised_mean + 0.033783) > 0.005:
    print(">>> WARNING: There is a large discrepancy! <<<")
    print("Possible causes:")
    print("  - show_all_returns uses log returns (continuously compounded)")
    print("  - The table shows arithmetic mean, but computed differently")
    print("  - The strategy return is defined as 1 + weight*spread, but the table might show excess over cash")

## Shiller + Fed + CMO Data: Gold since 1971 (65 years of data)

In [ ]:
col = {"CRUDE_DUBAI":"oil (spot)"}
for c in ["GOLD","COPPER"]:
    col[c] = c.lower()
dfcmo = read_shiller_cmo(col)
dfmetrics,filename = show_col_returns(col,show=True,excl=[])
print(dfmetrics.to_markdown(floatfmt=".2%"))


In [ ]:
col = {"CRUDE_DUBAI":"oil (spot)"}
for c in ["GOLD","COPPER","PLATINUM","SILVER"]:
    col[c] = c.lower()
dfmetrics,filename = show_col_returns(col,show=True,excl=[])
print(dfmetrics.to_markdown(floatfmt=".2%"))


## Shiller + Fed + CMO + Bloomberg Data: from 1991 (35y)

In [ ]:
def getbbdata(tick="CL"):
    dfcl = pd.read_csv(f"data/{tick.lower()}.csv")
    dfcl["date"] = pd.to_datetime(dfcl["date"])
    dfcl = dfcl.set_index("date")
    dfcl = dfcl.resample('M').last()
    dfcl = dfcl.reset_index()
    dfcl["date"] = dfcl["date"]+dt.timedelta(days=1)
    dfcl = dfcl.set_index("date")
    # Primary axis: CL1 and CL6
    fig, ax1 = plt.subplots(figsize=(10, 6))
    ax1.plot(dfcl[tick+"1"], label=tick+"1", color="tab:blue")
    ax1.plot(dfcl[tick+"6"], label=tick+"6", color="tab:blue", linestyle="--")
    ax1.set_ylabel("price")
    ax1.tick_params(axis='y', labelcolor="tab:blue")
    # Secondary axis: CL6 - CL1 (in gray)
    ax2 = ax1.twinx()
    ax2.plot(1 - dfcl[tick+"1"]/dfcl[tick+"6"], label=f"1 - {tick}1/{tick}6", color="gray", linestyle="--", alpha=0.8)
    ax2.set_ylabel("contango", color="gray")
    ax2.axhline(y=0,color="gray")
    ax2.tick_params(axis='y', labelcolor="gray")
    # Legend
    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left")
    plt.title(f"{tick} Price and Contango")
    plt.grid(True, alpha=0.3)
    plt.show()
    return dfcl

def addcontango(dfcl,tick):
    contango = (1+(dfcl[tick+"6"]-dfcl[tick+"1"])/dfcl[tick+"6"])**(1/5)-1-dfcl["TB3MS"]/12 # net carry
    totret   = dfcl[tick+"6"].pct_change()-contango.shift(1)+1#+(dfcl["tbTR"]-1) # excess return mu*_o=mu_o-mu_
    dfcl[f"{tickmap[tick]}CarryStratTR"] = 1+(totret-1)*np.where(contango<0.0,1,-1) 
    dfcl[f"{tickmap[tick]}TR"] = totret
    dfcl[tick+"contango"] = contango
    return dfcl
tickmap = {"CL":"oil","HG":"copper"}
tick="CL"
dfcl = getbbdata(tick).join(getbbdata(tick="HG"))
df = read_shiller_cmo({'GOLD': 'gold'})
dfcl = dfcl.join(df)

dfcl = addcontango(dfcl,tick).join(addcontango(dfcl,tick="HG"))


## Single Asset Strategies to outperform Long Only

### Oil Contango based Strategy
- most time is backwardation
- backwardation appears bullish, contango is bearish
- oil total return shows future total return, oil carry strategy goes long in backwardation, short in contango

In [ ]:
def contangocarrypnl(tick):
    x = (dfcl[tick+"contango"]).shift(1)
    y = dfcl[f"{tickmap[tick]}TR"]-1
    mask = ~(np.isnan(x) | np.isnan(y))
    x_clean = x[mask].values
    y_clean = y[mask].values
    coeffs = np.polyfit(x_clean, y_clean, 1)
    a, b = coeffs
    y_pred = a * x_clean + b
    errstd = np.std(y_clean - y_pred)
    plt.scatter(x, y, alpha=0.2)
    plt.plot(x_clean, np.polyval(coeffs, x_clean), 'r-', lw=2)
    plt.text(0.05, 0.9, f'y = {coeffs[0]:.4f}x + {coeffs[1]:.4f} + {errstd:.4f} e', 
             transform=plt.gca().transAxes, bbox=dict(facecolor='white', alpha=0.7))
    plt.axvline(x=-b/a,color="black")
    plt.xlabel(f"{tickmap[tick]} contango")
    plt.ylabel(f"6M {tickmap[tick]} future total return")
    plt.title(f"{tickmap[tick]} total return vs contango - pivot: {-b/a:.2%}")
    plt.show()
    print(a,b,errstd)
    plt.plot(np.cumprod(dfcl[f"{tickmap[tick]}CarryStratTR"]),label=f"{tickmap[tick]} carry strat")
    plt.plot(np.cumprod(dfcl[f"{tickmap[tick]}TR"]),label=f"long {tickmap[tick]} future total return")
    plt.yscale('log')
    plt.legend()
    plt.grid(True)

In [ ]:
contangocarrypnl(tick="CL")

In [ ]:
contangocarrypnl(tick="HG")

In [ ]:
print(show_returns(dfcl,filename=None,excl=["oilCarryStratTR","copperCarryStratTR"]).to_markdown(floatfmt=".2%"))

## Future roll impact on pnl for oil
- data from 1991 only
- impact of backwardation is massively positive
- be careful with oil front contract, it gets negative in mar 2020
- we can use oil future total return, or much better, oil carry strategy

In [ ]:
print(show_returns(dfcl,filename=None,excl=["oilTR","copperCarryStratTR"]).to_markdown(floatfmt=".2%"))

In [ ]:
print(show_returns(dfcl,filename=None,excl=["oilTR"]).to_markdown(floatfmt=".2%"))

### Rate study
- 10y bond is considered risky even though gov can print money because it bears rate risk
- 3m bond is considered risk free
- 10y vs 3m scatter plot shows correlation
- bond net return is flat vs 3m rate
- bond net return has slop vs 10y03m spread

In [ ]:
# Primary axis: CL1 and CL6
dfsh = read_shiller_out()
fig, ax1 = plt.subplots(figsize=(10, 6))

ax1.plot(np.cumproduct(1+dfsh["bondTR"]-dfsh["tbTR"]),color="tab:blue")
ax1.set_ylabel("net pnl")
ax1.tick_params(axis='y', labelcolor="tab:blue")

# Secondary axis: CL6 - CL1 (in gray)
ax2 = ax1.twinx()
ax2.plot(dfsh["Rate10y"], label="Rate10y", color="gray", linestyle="--", alpha=0.8)
ax2.plot(dfsh["TB3MS"], label="TB3MS", color="gray", linestyle=":", alpha=0.8)
ax2.plot(dfsh["Rate10y"]-dfsh["TB3MS"], label="spread", color="orange", linestyle=":", alpha=0.8)
ax2.axhline(y=0,color="orange")
ax2.set_ylabel("rate", color="gray")
ax2.tick_params(axis='y', labelcolor="gray")

# Legend
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left")

plt.title("Bond excess return and Rates")
plt.grid(True, alpha=0.3)
plt.show()

## Gold return
- gold does well when inflation fear is high: when bonds don't return much over tbills.

In [ ]:
x = (dfcmo["goldTR"]-dfcmo["tbTR"]).rolling(1).mean().shift(1)
y = dfcmo["goldTR"]-dfcmo["tbTR"]
mask = ~(np.isnan(x) | np.isnan(y))
x_clean = x[mask].values
y_clean = y[mask].values
coeffs = np.polyfit(x_clean, y_clean, 1)
a, b = coeffs
y_pred = a * x_clean + b
errstd = np.std(y_clean - y_pred)
goldcarrystrat = 1+(dfcmo["goldTR"]-dfcmo["tbTR"])*np.where(x>0,1,-1)
dfcmo["goldCarryStratTR"] = goldcarrystrat
dfmetrics,r,K = metrics_monthly_ret(dfcmo)
print(r,K)
print(dfmetrics.to_markdown(floatfmt=".2%"))
plt.plot(np.cumprod(goldcarrystrat),label="gold carry strat")
plt.plot(np.cumprod(1+(dfcmo["goldTR"]-dfcmo["tbTR"])),label="gold long")
plt.legend()
plt.yscale('log')
plt.grid(True)
plt.show()
plt.scatter(x, y, alpha=0.2)
plt.plot(x_clean, np.polyval(coeffs, x_clean), 'r-', lw=2)
plt.text(0.05, 0.9, f'y = {coeffs[0]:.4f}x + {coeffs[1]:.4f} + {errstd:.4f} e', 
         transform=plt.gca().transAxes, bbox=dict(facecolor='white', alpha=0.7))
plt.axvline(x=-b/a,color="black")
plt.xlabel("10y-3m spread")
plt.ylabel("gold net total return")
plt.title(f"Gold net return vs 10y-3m spread - pivot: {-b/a:.2%}")
plt.show()
print(a,b,errstd)

## Copper has econ PhD
- copper move are highly correlated to past 3m equity return

In [ ]:
x = (dfcl["eqTR"]-dfcl["tbTR"]).rolling(3).mean().shift(1)
y = dfcl["copperTR"]-dfcl["tbTR"]
mask = ~(np.isnan(x) | np.isnan(y))
x_clean = x[mask].values
y_clean = y[mask].values
coeffs = np.polyfit(x_clean, y_clean, 1)
a, b = coeffs
y_pred = a * x_clean + b
errstd = np.std(y_clean - y_pred)
plt.scatter(x, y, alpha=0.2)
plt.plot(x_clean, np.polyval(coeffs, x_clean), 'r-', lw=2)
plt.text(0.05, 0.9, f'y = {coeffs[0]:.4f}x + {coeffs[1]:.4f} + {errstd:.4f} e', 
         transform=plt.gca().transAxes, bbox=dict(facecolor='white', alpha=0.7))
plt.axvline(x=-b/a,color="black")
plt.xlabel("past 3m mean eq net return")
plt.ylabel("copper excess return")
plt.title(f"copper excess retun vs past eq excess return pivot: {-b/a:.2%}")
plt.show()
print(a,b,errstd)

## Eq future return
- appears to be strongly correlated to prev 8m perf

In [ ]:
x = (dfcl["eqTR"]-dfcl["tbTR"]).rolling(8).mean().shift(1)
y = dfcl["eqTR"]-dfcl["tbTR"]
mask = ~(np.isnan(x) | np.isnan(y))
x_clean = x[mask].values
y_clean = y[mask].values
coeffs = np.polyfit(x_clean, y_clean, 1)
a, b = coeffs
y_pred = a * x_clean + b
errstd = np.std(y_clean - y_pred)
plt.scatter(x, y, alpha=0.2)
plt.plot(x_clean, np.polyval(coeffs, x_clean), 'r-', lw=2)
plt.text(0.05, 0.9, f'y = {coeffs[0]:.4f}x + {coeffs[1]:.4f} + {errstd:.4f} e', 
         transform=plt.gca().transAxes, bbox=dict(facecolor='white', alpha=0.7))
plt.axvline(x=-b/a,color="black")
plt.xlabel("past 12m mean eq net return")
plt.ylabel("eq return")
plt.title(f"eq retun vs past eq return pivot: {-b/a:.2%}")
plt.show()
print(a,b,errstd)